# RQ1 sampling for open coding

Draws the samples the coders work on. Run `load_dataset.ipynb` first: its last cell saves
`rq1_frame.parquet`, which this notebook reads.

Two phases:

1. **Codebook rounds.** Small samples (default 50). Two coders code each round independently,
   resolve conflicts, update the codebook. Repeat with new pairs until a round adds no new codes.
2. **Measurement sample.** With the codebook frozen, code a fresh sample sized for a
   95% confidence level and 5% margin of error, stratified by transition type.

Nothing is ever drawn twice. Every draw is saved to `coding_rounds/`, and later draws skip any
pair already in a saved file.

**Coders work blind.** The coding sheet does not show the security label (`llm_transition`),
so it can't influence the codes. The label sits in a separate key file and is only joined back
after coding.

In [1]:
import math
from pathlib import Path

import pandas as pd
from IPython.display import display

FRAME_PATH = Path("rq1_frame.parquet")
CODING_DIR = Path("coding_rounds")
CODING_DIR.mkdir(exist_ok=True)

SEED_BASE = 2026   # change only if you restart sampling from scratch

frame = pd.read_parquet(FRAME_PATH)

frame["pair_id"] = (frame["skill_slug"] + "@" +
                    frame["prev_version"].astype(str) + "->" +
                    frame["version"].astype(str))
assert frame["pair_id"].is_unique, "pair_id is not unique, check the frame"

# Transition groups used for stratifying
GROUP = {"introduced": "introduced", "fixed": "fixed",
         "stayed clean": "no change", "stayed flagged": "no change"}
frame["stratum"] = frame["llm_transition"].map(GROUP)
frame = frame[frame["stratum"].notna()].copy()

print(f"{len(frame):,} pairs in the sampling frame, {frame['skill_slug'].nunique():,} skills\n")
display(pd.crosstab(frame["llm_transition"], frame["changelog_source"],
                    margins=True, margins_name="total"))

93,094 pairs in the sampling frame, 21,723 skills



changelog_source,auto,user,total
llm_transition,,,
fixed,2617,8164,10781
introduced,3301,11068,14369
stayed clean,10423,28053,38476
stayed flagged,8253,21215,29468
total,24594,68500,93094


In [2]:
# Columns the coders see, and columns kept back in the key file
SHEET_COLS = ["pair_id", "skill_slug", "prev_version", "version",
              "changelog_source", "changelog_text"]
KEY_COLS = ["pair_id", "llm_transition", "stratum", "pipeline_transition",
            "pipeline_changed", "model_changed", "changelog_source"]
CODER_COLS = ["coder1_code", "coder2_code", "agreed_code", "notes"]

def already_drawn():
    # Every pair_id in any saved key file, from any phase.
    files = list(CODING_DIR.glob("*_key.csv"))
    if not files:
        return set()
    return set(pd.concat(pd.read_csv(f, usecols=["pair_id"]) for f in files)["pair_id"])

def remaining_pool():
    return frame[~frame["pair_id"].isin(already_drawn())]

def save_draw(sample, name):
    sheet_path = CODING_DIR / f"{name}_sheet.csv"
    key_path = CODING_DIR / f"{name}_key.csv"
    if sheet_path.exists() or key_path.exists():
        raise FileExistsError(f"{name} already exists. Use a new name/round number; "
                              f"never overwrite a drawn sample.")
    # shuffle row order so coders don't see pairs grouped by stratum
    sample = sample.sample(frac=1, random_state=SEED_BASE)
    sheet = sample[[c for c in SHEET_COLS if c in sample.columns]].copy()
    for c in CODER_COLS:
        sheet[c] = ""
    sheet.to_csv(sheet_path, index=False)
    sample[[c for c in KEY_COLS if c in sample.columns]].to_csv(key_path, index=False)
    print(f"saved {len(sample)} pairs")
    print(f"  coding sheet (give to coders): {sheet_path}")
    print(f"  key (keep back until coding is done): {key_path}")
    print(f"  {len(remaining_pool()):,} pairs left in the pool")
    print("\nby stratum:")
    display(sample["stratum"].value_counts().rename("pairs").to_frame())

## Phase 1: codebook rounds

Each round draws `n` new pairs. With `balanced=True` the round is split evenly across
introduced / fixed / no change, so the coders see all kinds of transition while building
the codebook. Set it to `False` for a plain random draw.

Change `ROUND` each time. The cell refuses to overwrite a round that already exists.

In [15]:
ROUND = 8
N_PER_ROUND = 50
BALANCED = True

def draw_round(round_no, n=N_PER_ROUND, balanced=BALANCED):
    name = f"phase1_round{round_no:02d}"
    if (CODING_DIR / f"{name}_key.csv").exists():
        print(f"{name} was already drawn, nothing new drawn. Raise ROUND for a new round.")
        return pd.read_csv(CODING_DIR / f"{name}_sheet.csv")
    pool = remaining_pool()
    seed = SEED_BASE + round_no
    if balanced:
        groups = sorted(pool["stratum"].unique())
        per = {g: n // len(groups) + (1 if i < n % len(groups) else 0)
               for i, g in enumerate(groups)}
        parts = [g.sample(n=min(per[k], len(g)), random_state=seed)
                 for k, g in pool.groupby("stratum")]
        sample = pd.concat(parts)
    else:
        sample = pool.sample(n=n, random_state=seed)
    save_draw(sample, name)
    return sample

round_sample = draw_round(ROUND)

saved 50 pairs
  coding sheet (give to coders): coding_rounds\phase1_round08_sheet.csv
  key (keep back until coding is done): coding_rounds\phase1_round08_key.csv
  92,694 pairs left in the pool

by stratum:


,pairs
stratum,
introduced,17
fixed,17
no change,16


Diffs

In [24]:
import difflib
import gzip
import html
import io
import json
import re
import time
import zipfile

import requests
from tqdm.auto import tqdm

API = "https://clawhub.ai/api/v1"
FILES_CACHE = CODING_DIR / "_version_files"   # cached files of each downloaded version
MAX_FILE_BYTES = 1_000_000                    # bigger files are listed as changed but not diffed
SHOW_LINES_PER_FILE = 300                     # diff lines shown per file in the HTML view
session = requests.Session()


def _safe(value):
    return re.sub(r"[^A-Za-z0-9._-]", "_", str(value))


def fetch_zip(owner, slug, version, tries=5):
    status = "no response"
    for attempt in range(tries):
        try:
            r = session.get(f"{API}/download", timeout=60,
                            params={"slug": slug, "ownerHandle": owner, "version": version})
        except requests.RequestException:
            r = None
        if r is not None:
            status = r.status_code
            if r.status_code == 200:
                return r.content, "ok"
            if r.status_code == 404:
                return None, "not found"
        wait = 2 ** attempt
        if r is not None and r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 10 * (attempt + 1)))
        time.sleep(wait)
    return None, f"failed ({status})"


def version_files(skill_slug, version):
    """One version's files: ({path: text}, {path: crc of non-text files}, status). Cached."""
    owner, slug = skill_slug.split("/", 1)
    path = FILES_CACHE / _safe(owner) / f"{_safe(slug)}__{_safe(version)}.json.gz"
    if path.exists():
        with gzip.open(path, "rt", encoding="utf-8") as f:
            d = json.load(f)
        return d["text"], d["other"], "ok"
    blob, status = fetch_zip(owner, slug, version)
    if blob is None:
        return None, None, status
    text, other = {}, {}
    try:
        with zipfile.ZipFile(io.BytesIO(blob)) as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue
                if info.file_size <= MAX_FILE_BYTES:
                    try:
                        text[info.filename] = zf.read(info).decode("utf-8")
                        continue
                    except (UnicodeDecodeError, RuntimeError):
                        pass
                other[info.filename] = info.CRC
    except zipfile.BadZipFile:
        return None, None, "bad zip"
    # drop a top folder shared by every file (e.g. "my-skill/"), so paths match across versions
    names = list(text) + list(other)
    tops = {n.split("/", 1)[0] for n in names}
    if names and len(tops) == 1 and all("/" in n for n in names):
        cut = len(tops.pop()) + 1
        text = {n[cut:]: t for n, t in text.items()}
        other = {n[cut:]: c for n, c in other.items()}
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump({"text": text, "other": other}, f)
    return text, other, "ok"


def pair_diff(prev, curr):
    """Per changed file: dict(path, kind, added, removed, lines). SKILL.md first."""
    (pt, po), (ct, co) = prev, curr
    names = set(pt) | set(po) | set(ct) | set(co)
    IGNORE = {"skill-card.md", "_meta.json"}
    names = {n for n in names if n.split("/")[-1].lower() not in IGNORE}
    order = sorted(names, key=lambda n: (n.split("/")[-1].upper() != "SKILL.MD", n.lower()))
    out = []
    for name in order:
        in_prev, in_curr = name in pt or name in po, name in ct or name in co
        kind = "added" if not in_prev else "deleted" if not in_curr else "modified"
        if name in po or name in co:             # binary or too big on at least one side
            if po.get(name) is not None and po.get(name) == co.get(name):
                continue
            out.append({"path": name, "kind": kind + ", binary or too big, not shown",
                        "added": 0, "removed": 0, "lines": []})
            continue
        before, after = pt.get(name, "").splitlines(), ct.get(name, "").splitlines()
        if in_prev and in_curr and before == after:
            continue
        lines = list(difflib.unified_diff(before, after, f"a/{name}", f"b/{name}", lineterm=""))
        body = [l for l in lines[2:] if not l.startswith("@@")]
        out.append({"path": name, "kind": kind,
                    "added": sum(l.startswith("+") for l in body),
                    "removed": sum(l.startswith("-") for l in body),
                    "lines": lines[2:]})
    return out


CSS = """
body{font:15px/1.45 system-ui,Segoe UI,Arial,sans-serif;margin:0;background:#f6f5f2;color:#1e2a44}
main{max-width:1100px;margin:0 auto;padding:24px 16px}
h1{font-size:22px} .pair{background:#fff;border:1px solid #ddd8cc;border-radius:10px;padding:16px;margin:28px 0}
.pair h2{font-size:17px;margin:0 0 4px} .meta{color:#5b6474;font-size:13px}
.label{font-weight:600;margin:14px 0 6px} .cl{white-space:pre-wrap;background:#fbf8ef;border-left:4px solid #e0b450;padding:10px 12px;border-radius:4px}
details{border:1px solid #e5e1d8;border-radius:6px;margin:6px 0} summary{cursor:pointer;padding:6px 10px;font:13px ui-monospace,Consolas,monospace;background:#f3f1ec}
pre{margin:0;padding:6px 0;font:12.5px/1.4 ui-monospace,Consolas,monospace;white-space:pre-wrap;word-break:break-all}
pre span{display:block;padding:0 10px} .a{background:#e6f6ec} .r{background:#fbe9e9} .h{color:#7a8496;background:#f3f1ec}
.more{color:#5b6474;font-style:italic} .add{color:#1f7a5a} .del{color:#b42318}
table{border-collapse:collapse;font-size:13px} td,th{padding:3px 8px;border-bottom:1px solid #e5e1d8;text-align:left}
a{color:#2f5bd3}
"""


def _pair_html(n, row, files, status):
    esc = html.escape
    parts = [f'<section class="pair" id="p{n}"><h2>#{n} &nbsp;{esc(row.pair_id)}</h2>',
             f'<div class="meta">sheet row {n} (Excel row {n + 1})</div>',
             '<div class="label">Changelog (what the author says changed)</div>',
             f'<div class="cl">{esc(str(row.changelog_text))}</div>',
             '<div class="label">Diff (what actually changed)</div>']
    if files is None:
        parts.append(f'<p class="more">Diff not available: {esc(status)}</p>')
    elif not files:
        parts.append('<p class="more">No file changes between these two versions.</p>')
    for f in files or []:
        lines = f["lines"]
        shown = lines[:SHOW_LINES_PER_FILE]
        rows = []
        for l in shown:
            cls = "h" if l.startswith("@@") else "a" if l.startswith("+") else "r" if l.startswith("-") else ""
            rows.append(f'<span class="{cls}">{esc(l) or "&nbsp;"}</span>')
        if len(lines) > len(shown):
            rows.append(f'<span class="more">... {len(lines) - len(shown):,} more lines, '
                        f'see the full .diff file</span>')
        opened = " open" if len(files) <= 6 else ""
        parts.append(f'<details{opened}><summary>{esc(f["path"])} &nbsp;({esc(f["kind"])}) '
                     + ("" if not f["lines"] else
                        f'<span class="add">+{f["added"]}</span> <span class="del">-{f["removed"]}</span>')
                     + '</summary>'
                     + (f'<pre>{"".join(rows)}</pre>' if rows else "") + '</details>')
    parts.append("</section>")
    return "".join(parts)


def _read_sheet(name):
    path = CODING_DIR / f"{name}_sheet.csv"
    if not path.exists():
        raise FileNotFoundError(f"{name}_sheet.csv not found, did you run the draw?")
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")


def _pair_files(row, pause):
    """Changed files of one pair (None if a version is missing) and a status text."""
    pt, po, s1 = version_files(row.skill_slug, row.prev_version)
    time.sleep(pause)
    ct, co, s2 = version_files(row.skill_slug, row.version)
    time.sleep(pause)
    if pt is None or ct is None:
        return None, f"previous version: {s1}, this version: {s2}"
    return pair_diff((pt, po), (ct, co)), "ok"


def _stat_row(n, row, files, status):
    return {"row": n, "pair_id": row.pair_id, "diff_status": status,
            "files_changed": None if files is None else len(files),
            "lines_added": None if files is None else sum(f["added"] for f in files),
            "lines_removed": None if files is None else sum(f["removed"] for f in files),
            "changed_files": "" if files is None else "; ".join(f["path"] for f in files)}


def _write_view(title, stats, cards, out, note):
    index = "".join(
        f'<tr><td><a href="#p{r.row}">#{r.row}</a></td><td>{html.escape(r.pair_id)}</td>'
        f'<td>{"" if pd.isna(r.files_changed) else int(r.files_changed)}</td>'
        f'<td class="add">{"" if pd.isna(r.lines_added) else "+" + str(int(r.lines_added))}</td>'
        f'<td class="del">{"" if pd.isna(r.lines_removed) else "-" + str(int(r.lines_removed))}</td></tr>'
        for r in stats.itertuples())
    page = (f'<!doctype html><html><head><meta charset="utf-8"><title>{html.escape(title)} diffs</title>'
            f'<style>{CSS}</style></head><body><main><h1>{html.escape(title)}: changelog + diff</h1>'
            f'<p class="meta">{note} Green lines were added, red lines removed. '
            'Security labels are not shown here, so coding stays blind.</p>'
            '<table><tr><th>#</th><th>pair</th><th>files</th><th>added</th><th>removed</th></tr>'
            f'{index}</table>{"".join(cards)}</main></body></html>')
    out.write_text(page, encoding="utf-8")


def attach_diffs(name, pause=0.3, html=True):
    """Writes <name>_diffstats.csv and <name>_diffs/NNN.diff, plus <name>_diffs.html (coding view)
    if html=True. Never touches the coding sheet itself."""
    sheet = _read_sheet(name)
    full_dir = CODING_DIR / f"{name}_diffs"
    full_dir.mkdir(exist_ok=True)
    stats, cards = [], []
    for n, row in enumerate(tqdm(list(sheet.itertuples(index=False)), desc=name), start=1):
        files, status = _pair_files(row, pause)
        if files is not None:
            text = "\n".join(f'--- a/{f["path"]}\n+++ b/{f["path"]}\n' + "\n".join(f["lines"])
                             if f["lines"] else f'# {f["path"]}: {f["kind"]}' for f in files)
            (full_dir / f"{n:03d}.diff").write_text(text, encoding="utf-8")
        if html:
            cards.append(_pair_html(n, row, files, status))
        stats.append(_stat_row(n, row, files, status))
    stats = pd.DataFrame(stats)
    stats.to_csv(CODING_DIR / f"{name}_diffstats.csv", index=False, encoding="utf-8-sig")
    ok = (stats["diff_status"] == "ok").sum()
    print(f"{ok} of {len(stats)} pairs have a diff")
    if html:   # the html argument hides the html module here, so _write_view builds the page
        out = CODING_DIR / f"{name}_diffs.html"
        _write_view(name, stats, cards, out, "Same order as the coding sheet.")
        print(f"  open in a browser next to the sheet: {out}")
    print(f"  stats: {CODING_DIR / f'{name}_diffstats.csv'}   full diffs: {full_dir}")
    if ok < len(stats):
        display(stats[stats["diff_status"] != "ok"][["row", "pair_id", "diff_status"]])
    return stats


def attach_diffs_html(name, pair_ids, out_name, pause=0):
    """Writes <out_name>_diffs.html with only the given pairs of the <name> sheet, same layout.
    Reuses the cached version files. # is the row in the <name> sheet."""
    sheet = _read_sheet(name)
    wanted = set(pair_ids)
    missing = wanted - set(sheet["pair_id"])
    if missing:
        print(f"{len(missing)} pair_ids are not in the {name} sheet: {sorted(missing)}")
    stats, cards = [], []
    for n, row in enumerate(sheet.itertuples(index=False), start=1):
        if row.pair_id not in wanted:
            continue
        files, status = _pair_files(row, pause)
        cards.append(_pair_html(n, row, files, status))
        stats.append(_stat_row(n, row, files, status))
    stats = pd.DataFrame(stats, columns=["row", "pair_id", "diff_status", "files_changed",
                                         "lines_added", "lines_removed", "changed_files"])
    out = CODING_DIR / f"{out_name}_diffs.html"
    _write_view(out_name, stats, cards, out,
                f"Only the listed pairs, in {html.escape(name)} sheet order; # is the row in that sheet.")
    print(f"{len(stats)} pairs written. Open in a browser: {out}")
    return stats

In [16]:
# ===================== CODE CELL 2: build the view for a round =====================
# Run after each draw. Works for round 1 too, even though it is already coded.
diff_stats = attach_diffs(f"phase1_round{ROUND:02d}")

phase1_round08:   0%|          | 0/50 [00:00<?, ?it/s]

47 of 50 pairs have a diff
  open in a browser next to the sheet: coding_rounds\phase1_round08_diffs.html
  stats: coding_rounds\phase1_round08_diffstats.csv   full diffs: coding_rounds\phase1_round08_diffs


,row,pair_id,diff_status
12,13,securityqq/vargai@2.0.2->2.0.3,"previous version: not found, this version: not..."
41,42,heygen-com/hyperframes-animation@1.0.0->1.0.1,"previous version: not found, this version: not..."
43,44,viratkumar123/model-troubleshooter@1.0.0->1.0.1,"previous version: not found, this version: not..."


## Phase 2: measurement sample

Only run this once the codebook is frozen.

Sample size per stratum uses the standard formula with a finite population correction:
n0 = z² p(1-p) / e², n = n0 / (1 + (n0 - 1) / N), with z = 1.96, p = 0.5, e = 0.05.

- **Option A** (`PER_STRATUM = True`): a full 95/5 sample in each stratum. About 380 per
  stratum, so roughly 1,150 pairs.
- **Option B** (`PER_STRATUM = False`): one 95/5 sample of about 385, split evenly across
  strata. Cheaper, but each stratum's margin of error is wider. The cell prints it.

Phase 1 pairs are excluded automatically, since they shaped the codebook.

In [25]:
PER_STRATUM = True   # True = option A, False = option B
OVERSAMPLE = 1.15    # draw extra pairs per stratum; each size is multiplied, rounded up, capped at the pool
Z, P, E = 1.96, 0.5, 0.05

def sample_size(N, z=Z, p=P, e=E):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))

def moe(n, N, z=Z, p=P):
    fpc = math.sqrt((N - n) / (N - 1)) if N > 1 else 0
    return z * math.sqrt(p * (1 - p) / n) * fpc

pool = remaining_pool()
groups = dict(tuple(pool.groupby("stratum")))

if PER_STRATUM:
    sizes = {k: sample_size(len(g)) for k, g in groups.items()}
else:
    total = sample_size(len(pool))
    sizes = {k: math.ceil(total / len(groups)) for k in groups}

# round() first so float error (e.g. 23.000000000000004) can't add a pair
draw_sizes = {k: min(math.ceil(round(sizes[k] * OVERSAMPLE, 9)), len(g)) for k, g in groups.items()}

plan = pd.DataFrame([
    {"stratum": k, "pool": len(g),
     "sample": min(sizes[k], len(g)), "margin_of_error": moe(min(sizes[k], len(g)), len(g)),
     "sample_oversampled": draw_sizes[k], "margin_of_error_oversampled": moe(draw_sizes[k], len(g))}
    for k, g in groups.items()
])
print("Option", "A (95/5 per stratum)" if PER_STRATUM else "B (one 95/5 sample, split evenly)")
print(f"oversampling x{OVERSAMPLE}, rounded up, capped at the pool size")
display(plan.style.format({"margin_of_error": "{:.1%}", "margin_of_error_oversampled": "{:.1%}"}))
print(f"total pairs without oversampling: {int(plan['sample'].sum()):,}")
print(f"total pairs with oversampling:    {int(plan['sample_oversampled'].sum()):,}  (this is what the draw uses)")

Option A (95/5 per stratum)
oversampling x1.15, rounded up, capped at the pool size


,stratum,pool,sample,margin_of_error,sample_oversampled,margin_of_error_oversampled
0,fixed,10645,371,5.0%,427,4.6%
1,introduced,14233,375,5.0%,432,4.6%
2,no change,67816,383,5.0%,441,4.7%


total pairs without oversampling: 1,129
total pairs with oversampling:    1,300  (this is what the draw uses)


In [27]:
# Draw it. Run only after checking the plan above.
DRAW_PHASE2 = True
MAX_PER_OWNER = 2   # owner = part of skill_slug before "/"

def draw_capped(groups, n_per_stratum, seed, cap=MAX_PER_OWNER):
    """Per stratum: shuffle with the seed and take pairs in that order, skipping any pair whose
    owner already has `cap` pairs in the sample (counted across all strata). Skipped pairs are
    replaced by the next ones in the same stratum. The shuffle order matches g.sample(n=..., seed),
    so without the cap this gives the same draw."""
    owner_n, parts, report = {}, [], []
    for k in sorted(groups):
        g = groups[k].sample(frac=1, random_state=seed)
        want = n_per_stratum[k]
        owners = g["skill_slug"].str.split("/", n=1).str[0]
        keep, skipped = [], 0
        for pos, (idx, owner) in enumerate(owners.items()):
            if len(keep) == want:
                break
            if owner_n.get(owner, 0) >= cap:
                skipped += pos < want   # was in the uncapped draw, replaced by a later pair
                continue
            owner_n[owner] = owner_n.get(owner, 0) + 1
            keep.append(idx)
        parts.append(g.loc[keep])
        report.append({"stratum": k, "wanted": want, "drawn": len(keep),
                       "replaced_by_cap": skipped, "short": want - len(keep)})
    return pd.concat(parts), pd.DataFrame(report)

if (CODING_DIR / "phase2_key.csv").exists():
    print("phase 2 was already drawn, nothing new drawn.")
elif DRAW_PHASE2:
    phase2, cap_report = draw_capped(groups, draw_sizes, SEED_BASE + 999)
    print(f"owner cap ({MAX_PER_OWNER} per owner) affected {int(cap_report['replaced_by_cap'].sum())} pairs, "
          "replaced from the same stratum where possible")
    display(cap_report)
    if cap_report["short"].any():
        print("some strata ran out of pairs under the cap, see 'short'")
    save_draw(phase2, "phase2")
else:
    print("set DRAW_PHASE2 = True to draw")

owner cap (2 per owner) affected 345 pairs, replaced from the same stratum where possible


,stratum,wanted,drawn,replaced_by_cap,short
0,fixed,427,427,58,0
1,introduced,432,432,116,0
2,no change,441,441,171,0


saved 1300 pairs
  coding sheet (give to coders): coding_rounds\phase2_sheet.csv
  key (keep back until coding is done): coding_rounds\phase2_key.csv
  91,394 pairs left in the pool

by stratum:


,pairs
stratum,
no change,441
introduced,432
fixed,427


In [28]:
# Diffs for the Phase 2 sample: .diff files and diffstats only, no HTML view
diff_stats = attach_diffs("phase2", html=False)

phase2:   0%|          | 0/1300 [00:00<?, ?it/s]

1289 of 1300 pairs have a diff
  stats: coding_rounds\phase2_diffstats.csv   full diffs: coding_rounds\phase2_diffs


,row,pair_id,diff_status
569,570,heygen-com/general-video@1.0.7->1.0.8,"previous version: not found, this version: not..."
576,577,agenticweb4/huawei-cloud-billing-scout@1.0.0->...,"previous version: failed (410), this version: ok"
594,595,degen0root/panchanga-api@3.0.2->3.1.0,"previous version: ok, this version: failed (403)"
735,736,agenticweb4/huawei-cloud-billing-scout@2.1.0->...,"previous version: failed (410), this version: ..."
838,839,wxt-ai/routing-contact-workbench@1.0.4->1.0.7,"previous version: not found, this version: not..."
899,900,jackortel-gif/freelance-clearing@1.0.0->1.0.3,"previous version: not found, this version: not..."
1179,1180,macd2/agent-crypto-wallet-depricated@1.9.1->1.9.9,"previous version: not found, this version: not..."
1190,1191,drumrobot/commit-tidy@0.4.2->0.4.3,"previous version: not found, this version: not..."
1201,1202,bartelmost/agentshield-audit@1.0.15->1.0.16,"previous version: not found, this version: not..."
1210,1211,wxt-ai/project-code-notes-workbench@1.0.0->1.0.1,"previous version: not found, this version: not..."


## After Phase 2 coding: join the codes back to the labels

Only now are the coded sheets joined to the key files, so the security label meets the code
for the first time here.

In [ ]:
P2_DIR = Path("phase2_coding")
P2_CODES = P2_DIR / "phase2_codes.csv"
P2_SPOT = P2_DIR / "phase2_spotcheck.csv"

# stated_motive groups from the Phase 1 codebook (open_coding/themes.md): keyword match, first group wins
MOTIVE_GROUPS = [
    ('Respond to registry scanner or audit findings (incl. reducing false positives)',
     ['security-audit', 'security-scan', 'security warnings', 'ClawScan', 'suspicious', 'false-positive', 'false positive', 'SkillSpector', 'Registry Mismatch', 'security concern', 'ClawHub feedback', 'security wording', 'triggered security scanner', 'sensitive path warnings', 'trust cleanup']),
    ('Security hardening / prevent harm',
     ['security', 'hardening', 'hardened', 'exfiltration', 'injection', 'disclose local files', 'plaintext token', 'never hardcoded', 'propagation', 'pre-created by another user', 'no longer stored on disk', 'accidental execution', 'risk-reducing', 'Chrome windows']),
    ('Fix a defect or improve reliability/compatibility/performance',
     ['import failure', 'torn JSON', 'recovers', 'understated', '--live', 'UnicodeEncodeError', 'already sent', 'stale', 'never surfaced', 'ABI', 'silently skipped', '403s', 'unnecessary API calls', 'slow URL validation', 'reliability', 'compatibility', 'survive skill upgrades']),
    ('Registry or metadata compatibility',
     ['registry parses', 'align metadata', 'unsupported skill frontmatter', 'registry/install', 'match real requirements']),
    ('Simplify setup or usage',
     ['cleaner setup', 'simplify configuration', 'simplifying', 'simpler setup', 'lightweight', 'brevity']),
    ('Consistency of formatting',
     ['consistency']),
    ('Accurate or honest description',
     ['honest', 'duplicate premium', 'credential handling contradiction']),
    ('Discoverability and traffic tracking',
     ['discoverability', 'traffic source', 'UTM truncation']),
    ('Product or business structure',
     ['paid features', 'canonical maintained', 'external wallet'])
]

def motive_group(text):
    t = text.lower()
    for name, words in MOTIVE_GROUPS:
        if any(w.lower() in t for w in words):
            return name
    return "other"

def pair_table(df, col, n_pairs, title):
    """Pairs per value of col x stratum. A pair counts once per value, even with several key points."""
    pairs = df[["pair_id", "stratum", col]].drop_duplicates()
    counts = pd.crosstab(pairs[col], pairs["stratum"])
    counts["total"] = counts.sum(axis=1)
    shares = counts / n_pairs[counts.columns]   # share of the included pairs in each stratum
    print(f"{title} (pairs):")
    display(counts)
    print(f"{title} (share of pairs within each stratum):")
    display(shares.style.format("{:.1%}"))

if P2_CODES.exists():
    codes = pd.read_csv(P2_CODES, dtype=str).fillna("")
    key = pd.read_csv(CODING_DIR / "phase2_key.csv", dtype=str)
    codes = codes.merge(key[["pair_id", "stratum"]], on="pair_id", how="left")
    if codes["stratum"].isna().any():
        print(f"warning: {codes.loc[codes['stratum'].isna(), 'pair_id'].nunique()} coded pairs are not in phase2_key.csv")
    print(f"{codes['pair_id'].nunique():,} of {len(key):,} phase 2 pairs coded, {len(codes):,} key points\n")

    flags = codes.groupby("pair_id").agg(stratum=("stratum", "first"),
                                          no_author_change=("no_author_change", "first"),
                                          diff_unavailable=("diff_unavailable", "first"),
                                          changelog_mismatch=("changelog_mismatch", "first")).reset_index()
    flags["excluded"] = ""
    flags.loc[flags["no_author_change"] == "yes", "excluded"] = "no author change"
    flags.loc[flags["diff_unavailable"] == "yes", "excluded"] = "diff unavailable"
    excl = flags[flags["excluded"] != ""]
    print(f"excluded {len(excl):,} pairs (per stratum):")
    display(pd.crosstab(excl["excluded"], excl["stratum"], margins=True, margins_name="total"))

    pairs = flags[flags["excluded"] == ""]
    kp = codes[codes["pair_id"].isin(pairs["pair_id"])]
    n_pairs = pairs["stratum"].value_counts()
    n_pairs["total"] = len(pairs)
    print("included pairs per stratum:")
    display(n_pairs.rename("pairs").to_frame())

    pair_table(kp, "theme", n_pairs, "theme x stratum")
    pair_table(kp, "subtheme", n_pairs, "subtheme x stratum")

    # a pair counts for a direction within a theme if any of its key points in that theme has it
    d = kp[["pair_id", "stratum", "theme", "direction"]].drop_duplicates()
    print("direction x theme x stratum (pairs):")
    display(pd.crosstab([d["theme"], d["direction"]], d["stratum"], margins=True, margins_name="total"))

    pair_table(pairs, "changelog_mismatch", n_pairs, "changelog_mismatch x stratum")

    m = kp[kp["stated_motive"].str.strip() != ""]
    m = m.assign(motive_group=m["stated_motive"].map(motive_group))
    print(f"{m['pair_id'].nunique():,} included pairs state a motive (a pair can be in more than one group)")
    pair_table(m, "motive_group", n_pairs, "stated_motive groups x stratum")

    if P2_SPOT.exists():
        spot = pd.read_csv(P2_SPOT, dtype=str).fillna("")
        spot["coder_verdict"] = spot["coder_verdict"].str.strip().str.lower()
        done = spot[spot["coder_verdict"] != ""]
        if done.empty:
            print("spot-check: verdict columns not filled yet")
        else:
            done = done.assign(agree=done["coder_verdict"] == "agree")
            by_pair = done.groupby("pair_id")["agree"].all()
            print(f"spot-check: {len(done)} of {len(spot)} key points have a verdict, "
                  f"agree on {done['agree'].mean():.1%} of key points; "
                  f"{by_pair.mean():.1%} of {len(by_pair)} pairs agreed on every key point")
            display(done.groupby("subtheme")["agree"].agg(key_points="size", agree="mean")
                    .style.format({"agree": "{:.1%}"}))
            dis = done[~done["agree"]]
            if len(dis):
                print("disagreements:")
                display(dis[["pair_id", "key_point_no", "subtheme", "coder_subtheme", "coder_note"]])
    else:
        print("no phase 2 spot-check file yet")
else:
    print("no phase2_codes.csv yet")

In [ ]:
# HTML view of only the Phase 2 spot-check pairs, for the coders
spotcheck = pd.read_csv("phase2_coding/phase2_spotcheck.csv", encoding="utf-8-sig", dtype=str)
spot_stats = attach_diffs_html("phase2", spotcheck["pair_id"].unique().tolist(), "phase2_spotcheck")